# LionChief Plus Locomotives — Dataset Inspection & EDA

Quick data-quality pass over the scraped locomotive image dataset before using it to train an image classifier.

This notebook checks:
- Folder / class structure (one folder per SKU)
- How many images per SKU (class balance)
- Image formats, sizes, and aspect ratios
- Corrupt or unreadable images
- Duplicate images
- A visual sample grid to eyeball image quality

Dataset path: `/kaggle/input/datasets/datascientist97/lionchief-plus-locomotives-product-images-by-sku/lionel_dataset`

In [ ]:
import os
import hashlib
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATASET_DIR = "/kaggle/input/datasets/datascientist97/lionchief-plus-locomotives-product-images-by-sku/lionel_dataset"

assert os.path.isdir(DATASET_DIR), f"Dataset path not found: {DATASET_DIR}"
print("Dataset root:", DATASET_DIR)
print("Top-level entries:", len(os.listdir(DATASET_DIR)))

## 1. Walk the folder structure

Each subfolder is expected to be one SKU, containing 1+ product photos of that locomotive.

In [ ]:
VALID_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

records = []  # (sku, filename, full_path)

for sku in sorted(os.listdir(DATASET_DIR)):
    sku_path = os.path.join(DATASET_DIR, sku)
    if not os.path.isdir(sku_path):
        continue
    for fname in sorted(os.listdir(sku_path)):
        ext = os.path.splitext(fname)[1].lower()
        if ext in VALID_EXTS:
            records.append((sku, fname, os.path.join(sku_path, fname)))

df = pd.DataFrame(records, columns=["sku", "filename", "path"])
print(f"Total SKU folders: {df['sku'].nunique()}")
print(f"Total image files: {len(df)}")
df.head()

## 2. Class balance — images per SKU

This matters a lot for training: SKUs with only 1 photo will train noticeably weaker than ones with several angles.

In [ ]:
counts = df.groupby("sku").size().sort_values(ascending=False)

print("Images-per-SKU summary:")
print(counts.describe())
print()
print("Distribution of image counts:")
print(counts.value_counts().sort_index())

plt.figure(figsize=(8, 4))
counts.value_counts().sort_index().plot(kind="bar")
plt.title("Number of SKUs by image count")
plt.xlabel("Images per SKU")
plt.ylabel("Number of SKUs")
plt.tight_layout()
plt.show()

In [ ]:
single_image_skus = counts[counts == 1]
print(f"SKUs with only 1 image: {len(single_image_skus)} ({len(single_image_skus)/len(counts)*100:.1f}% of all SKUs)")
print("\nThese are the weakest training candidates — consider flagging for reshoot/more photos:")
single_image_skus.head(20)

## 3. Corrupt / unreadable image check

Opens and verifies every image with Pillow. Anything that fails should be excluded from training rather than silently breaking the pipeline later.

In [ ]:
corrupt_files = []
image_dims = []
image_formats = []
file_sizes_kb = []

for _, row in df.iterrows():
    path = row["path"]
    try:
        with Image.open(path) as img:
            img.verify()
        # re-open after verify() (verify() invalidates the file handle)
        with Image.open(path) as img:
            image_dims.append(img.size)
            image_formats.append(img.format)
        file_sizes_kb.append(os.path.getsize(path) / 1024)
    except Exception as e:
        corrupt_files.append((path, str(e)))

print(f"Checked {len(df)} images")
print(f"Corrupt/unreadable: {len(corrupt_files)}")

if corrupt_files:
    print("\nCorrupt files found:")
    for path, err in corrupt_files:
        print(f"  {path} — {err}")
else:
    print("✅ No corrupt images found.")

## 4. Image formats, dimensions, and file sizes

In [ ]:
print("Format breakdown:")
print(Counter(image_formats))

widths = [d[0] for d in image_dims]
heights = [d[1] for d in image_dims]
aspect_ratios = [w / h for w, h in image_dims]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(widths, bins=30)
axes[0].set_title("Image width distribution")
axes[0].set_xlabel("Width (px)")

axes[1].hist(heights, bins=30)
axes[1].set_title("Image height distribution")
axes[1].set_xlabel("Height (px)")

axes[2].hist(aspect_ratios, bins=30)
axes[2].set_title("Aspect ratio distribution (w/h)")
axes[2].set_xlabel("Aspect ratio")

plt.tight_layout()
plt.show()

print(f"\nMedian dimensions: {int(pd.Series(widths).median())} x {int(pd.Series(heights).median())}")
print(f"Min dimensions: {min(widths)} x {min(heights)}")
print(f"Max dimensions: {max(widths)} x {max(heights)}")
print(f"\nFile size (KB) — median: {pd.Series(file_sizes_kb).median():.1f}, min: {min(file_sizes_kb):.1f}, max: {max(file_sizes_kb):.1f}")

## 5. Duplicate image detection

Uses an MD5 hash of each file's raw bytes to catch exact duplicates — e.g. the same product photo accidentally saved under two different SKU folders, which would quietly corrupt training labels.

In [ ]:
def file_md5(path, chunk_size=8192):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

hash_to_paths = defaultdict(list)
for _, row in df.iterrows():
    try:
        h = file_md5(row["path"])
        hash_to_paths[h].append((row["sku"], row["path"]))
    except Exception:
        continue

duplicate_groups = {h: paths for h, paths in hash_to_paths.items() if len(paths) > 1}

print(f"Duplicate groups found: {len(duplicate_groups)}")

cross_sku_dupes = 0
for h, paths in duplicate_groups.items():
    skus_involved = set(p[0] for p in paths)
    if len(skus_involved) > 1:
        cross_sku_dupes += 1
        print(f"\n⚠️ Same image appears under multiple SKUs: {skus_involved}")
        for sku, path in paths:
            print(f"   {sku}: {path}")

if cross_sku_dupes == 0:
    print("✅ No exact-duplicate images shared across different SKUs.")

## 6. Visual sample grid

Random sample of images to eyeball quality — background consistency, framing, lighting.

In [ ]:
sample_df = df.sample(n=min(16, len(df)), random_state=42)

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
for ax, (_, row) in zip(axes.flat, sample_df.iterrows()):
    try:
        with Image.open(row["path"]) as img:
            ax.imshow(img)
        ax.set_title(row["sku"], fontsize=9)
    except Exception:
        ax.set_title(f"{row['sku']} (failed to load)", fontsize=9, color="red")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7. Summary

Quick recap of what to act on before training.

In [ ]:
print("===== DATASET SUMMARY =====")
print(f"Total SKUs (classes):        {df['sku'].nunique()}")
print(f"Total images:                {len(df)}")
print(f"Avg images per SKU:          {len(df) / df['sku'].nunique():.2f}")
print(f"SKUs with only 1 image:      {len(single_image_skus)}")
print(f"Corrupt/unreadable images:   {len(corrupt_files)}")
print(f"Cross-SKU duplicate images:  {cross_sku_dupes}")
print(f"Image formats present:       {dict(Counter(image_formats))}")
print()
print("Recommended next steps:")
if corrupt_files:
    print("  - Remove or re-download the corrupt image(s) listed in section 3.")
if cross_sku_dupes:
    print("  - Investigate cross-SKU duplicates in section 5 — likely a labeling error.")
if len(single_image_skus) > 0:
    print(f"  - {len(single_image_skus)} SKUs have only 1 training image — expect weaker accuracy on these until more angles are added.")
print("  - Dataset otherwise looks ready for tagging/upload into Azure Custom Vision (or similar).")